In [0]:
pip install pls_common_data_store

In [0]:
import os 
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import current_timestamp
import datetime as dt
from datetime import timedelta, datetime, date
from dateutil.relativedelta import relativedelta
import time 
from kayday import KrogerDate, DateRange
from effodata import ACDS, golden_rules, Joiner, Equality
from pls_common_data_store import pls_data_store
import seg
from kpi_metrics import KPI, AliasMetric, CustomMetric, AliasGroupby, Rollup, Cube, available_metrics, get_metrics
from effodata import ACDS, golden_rules, Joiner, Sifter, Equality
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
from statsmodels.stats.weightstats import CompareMeans, DescrStatsW
from pls_common_data_store import pls_data_store

spark = SparkSession.builder.getOrCreate()

In [0]:
# ACDS transactions table
acds_sample = ACDS(use_sample_mart = False)
start = (datetime.today() - relativedelta(months=24)).strftime('%Y%m%d')
end = datetime.today().strftime('%Y%m%d')
acds_transactions = acds_sample.get_transactions(start_date = start, end_date = end, join_with = ["products", "households"])

#### KPF Gift Card Segment HHs
- Pulling current segmentations, as well as historical segmentations.
- We can use specific time anchors to identify new 3p Gift HHs per fiscal month, and identify trends with features.

In [0]:
# KPF Gift Card Baseball Segmentation HHs - CURRENT

# KPF 3rd Pty Giftcard Catalog Link: https://adb-291758323461480.0.azuredatabricks.net/explore/data/segmentations_prd/kpf_third_party_giftcard?o=291758323461480&activeListType=TABLE

gift_hh_segment_df = spark.read.parquet(f"abfss://kpf@sa8451cpkpfprd.dfs.core.windows.net/segments/current/tpg/gift_card_segment_CURRENT")


filtered_gift_hh_segment_df = gift_hh_segment_df.filter(
    gift_hh_segment_df.gcs_seg_desc.isin(
        "ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS"
    )
)

display(filtered_gift_hh_segment_df)

distinct_fiscal_weeks = [row.fiscal_week for row in filtered_gift_hh_segment_df.select("fiscal_week").distinct().collect()]
print(distinct_fiscal_weeks)

In [0]:
# KPF Gift Card Baseball Segmentation HHs - OLD

gift_hh_segment_df_old = spark.read.parquet(f"abfss://kpf@sa8451cpkpfprd.dfs.core.windows.net/segments/current/tpg/gift_card_segment_HISTORICAL")

#filtered_gift_hh_segment_df_old = gift_hh_segment_df_old.filter(
#    gift_hh_segment_df_old.gcs_seg_desc.isin(
#        "ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS"
#    )
#)

display(gift_hh_segment_df_old)

distinct_fiscal_weeks = [row.fiscal_week for row in gift_hh_segment_df_old.select("fiscal_week").distinct().orderBy("fiscal_week").collect()]
print(distinct_fiscal_weeks)

#### ACDS Transactions during Gifting Seasons
- Can identify spending trends during peak KPF seasons. For example, identify trends of HHs who have recently become a 3P gift buyer at a certain time frame (for example, ehhns who were not a 3P Gift Buyer in Dec 2025 but became one in 2026 due to 2025 spending)
- Can do parametric test to determine if there is a significant difference in spending habits of HHs during certain kpf-popular seasons (and if true, then would be a good feature)

##### Holiday Timeframe
- Taking into account winter, new years, and holiday campaign timeframes (Period 12 - 13 spending) -> Period 1 baseball segment

In [0]:
# Obtaining HHs who are NEWLY converted 3P Gift Buyers at a certain date (Not Gift Buyer in 2025 Nov, but were in Dec )
# Used this time-period for diagnostics to line up with 2026 holiday xcm time

best_baseball_segments = ["ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS"]

# 2025 P12 + 13 to account for holiday + new year offers
gift_hh_segment_df_old_2025_12_04 = gift_hh_segment_df_old.filter(
    (f.col("fiscal_week") == "20251204") & (f.col("gcs_seg_desc").isin(best_baseball_segments))
).select("ehhn", "gcs_seg_desc")

gift_hh_segment_df_old_2025_11_04 = gift_hh_segment_df_old.filter(
    f.col("fiscal_week") == "20251104"
).select("ehhn")

new_3p_gift_households = gift_hh_segment_df_old_2025_12_04.join(
    gift_hh_segment_df_old_2025_11_04, on="ehhn", how="left_anti"
).dropDuplicates(["ehhn"])

display(new_3p_gift_households)

In [0]:
# Validation of "new HHs". If they were gift card buyers over a yr ago, means they are still new HHs.

gift_hh_segment_df_older_than_2025_nov = gift_hh_segment_df_old.filter(f.col("fiscal_week") <= "20251104")

ehhn_in_older = gift_hh_segment_df_older_than_2025_nov.join(
    new_3p_gift_households.select("ehhn"), on="ehhn", how="inner"
)
# showing the last time ehhns were a gift card buyer (if they ever were) .. should be over 52 weeks ago
display(ehhn_in_older)

In [0]:
# ACDS transactions table
acds_sample = ACDS(use_sample_mart = True)
start = (datetime.today() - relativedelta(months=12)).strftime('%Y%m%d')
end = datetime.today().strftime('%Y%m%d')
acds_transactions = acds_sample.get_transactions(start_date = start, end_date = end, join_with = ["products", "households"])
acds_transactions.limit(5).display()

In [0]:
# Filter to ACDS transactions between 20251204 and 20260104 (this time anchor will be different depending on the timeframe we choose above)
# Filter to ONLY HHs who are newly converted 3P Gift Buyers. We will compare this with HHs in this timeframe who are NOT gift buyers

acds_transactions_per_ehhn_per_date = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

acds_transactions_per_ehhn_per_date_filtered = acds_transactions_per_ehhn_per_date.filter(
    (f.col("ehhn").isin(new_3p_gift_households.select("ehhn"))) &
    # Date here, not period
    (f.col("trn_dt") >= "20251108") & (f.col("trn_dt") <= "20251205")
)

acds_transactions_per_ehhn_per_date_filtered.display()

In [0]:
# Grouped HHs by Avg spend per trip for only HHs who became 3p gift buyers
avg_spend_per_date_per_hh_soon_to_be_gift_buyer = acds_transactions_per_ehhn_per_date_filtered.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_soon_to_be_gift_buyer)

In [0]:
# Checking variance
variance_soon_to_be_gift_buyers = (
    avg_spend_per_date_per_hh_soon_to_be_gift_buyer.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_soon_to_be_gift_buyers.display()

In [0]:
# Filter to ACDS transactions between 20251204 and 20260104 (this time anchor will be different depending on the timeframe we choose above)
# Filter to HHs who will NOT become 3p gift buyers or are not in gift_hh_segment_df_old
# TLDR: Control Group. Remove all current gift card shoppers as well as HHs who will become a gift card shopper

acds_transactions_per_ehhn_per_date = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

# these are HHs who are going to become 3p gift buyers in 20260104 (as a result of their habits in 20251204), as well as HHs who are already gift card buyers

# remove any hh who has touched a gift card in the last 52 wks + hhs soon to be new 3p gift hhs 
relevant_gift_hh_segment_df_old = gift_hh_segment_df_old.filter(f.col("fiscal_week") >= "20241108")
ehhn_exclude = new_3p_gift_households.select("ehhn").union(relevant_gift_hh_segment_df_old.select("ehhn")).distinct()


acds_transactions_per_ehhn_per_date_filtered = acds_transactions_per_ehhn_per_date.join(
    ehhn_exclude, on="ehhn", how="left_anti"
).filter(
    (f.col("trn_dt") >= "20251108") & (f.col("trn_dt") <= "20251205")
)

acds_transactions_per_ehhn_per_date_filtered.display()

In [0]:
# Grouped HHs by Avg spend per trip for non gift hh
avg_spend_per_date_per_hh_not_gift_buyer = acds_transactions_per_ehhn_per_date_filtered.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_not_gift_buyer)

In [0]:
# Checking variance 
variance_non_gift_buyer = (
    avg_spend_per_date_per_hh_not_gift_buyer.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_non_gift_buyer.display()

In [0]:
# Parametric test to determine: is there a difference in spending between HHs who will become 3p gift buyers and HHs who will not
# during peak KPF seasons? (We are testing with Holiday as per 20251204 - 20260104, also do for valentines day, mothers/fathers day)

# TY genie
gift_buyer_spend_A = avg_spend_per_date_per_hh_soon_to_be_gift_buyer.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]
nongift_buyer_spend_B = avg_spend_per_date_per_hh_not_gift_buyer.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]

t_stat, p_val = stats.ttest_ind(gift_buyer_spend_A, nongift_buyer_spend_B, equal_var=False)

mean_test = np.mean(gift_buyer_spend_A)
mean_control = np.mean(nongift_buyer_spend_B)
raw_lift = mean_test - mean_control

# Using 95% CI
compare_means_gift_buyer_vs_non_gift_buyer = CompareMeans(DescrStatsW(gift_buyer_spend_A), DescrStatsW(nongift_buyer_spend_B))
lower_ci, upper_ci = compare_means_gift_buyer_vs_non_gift_buyer.tconfint_diff(alpha=0.05, usevar='unequal')

print(
    f"Test Mean Spend: ${mean_test:.2f} | "
    f"Control Mean Spend: ${mean_control:.2f} | "
    f"Observed Lift: {raw_lift:+.2f} per trip | "
    f"P-value: {p_val:.6f} | "
    f"95% Confidence Interval for Lift: [${lower_ci:.2f}, ${upper_ci:.2f}]"
)

# if p-value is less than 0.05 then difference in spend between soon-to-be gift buyer and non-gift buyer in time window is stat sig
# Results show that HHs who were about to become 3p gift buyers spent (statistically) significantly more than HHs who were not about to become 3p gift buyers in holiday gifting season.

##### Easter + Mother's Day + Father's Day + Summer Time Frame 
- In FY2026, we wont split these up. This is just to see whether transaction behaviors differ from soon-to-be gift hhs vs non gift hhs, and whether this would be a good feature for the LA model.

In [0]:
# Obtaining HHs who are NEWLY converted 3P Gift Buyers at a certain date (Not Gift Buyer in 2025 April/May, but is in 2025 June, as well as not a gift buyer in 2025 June, but is in 2025 July)

# Mothers Day (p4) ---------------------------------------------------------------------------------
best_baseball_segments = ["ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS", "BENCH WARMERS", "UTLITY PLAYERS"]

gift_hh_segment_df_old_20250504 = gift_hh_segment_df_old.filter(
    (f.col("fiscal_week") == "20250504") & (f.col("gcs_seg_desc").isin(best_baseball_segments))
).select("ehhn", "gcs_seg_desc")

gift_hh_segment_df_old_20250404_base = gift_hh_segment_df_old.filter(
    f.col("fiscal_week") == "20250404"
).select("ehhn")

new_3p_gift_households_mothers_day = gift_hh_segment_df_old_20250504.join(
    gift_hh_segment_df_old_20250404_base, on="ehhn", how="left_anti"
).dropDuplicates(["ehhn"])

display(new_3p_gift_households_mothers_day)

# Fathers Day (p5) ---------------------------------------------------------------------------------

gift_hh_segment_df_old_20250604 = gift_hh_segment_df_old.filter(
    (f.col("fiscal_week") == "20250604") & (f.col("gcs_seg_desc").isin(best_baseball_segments))
).select("ehhn", "gcs_seg_desc")

gift_hh_segment_df_old_20250504_base = gift_hh_segment_df_old.filter(
    f.col("fiscal_week") == "20250504"
).select("ehhn")

new_3p_gift_households_fathers_day = gift_hh_segment_df_old_20250604.join(
    gift_hh_segment_df_old_20250504_base, on="ehhn", how="left_anti"
).dropDuplicates(["ehhn"])

display(new_3p_gift_households_fathers_day)

In [0]:
# Validation of "new HHs". If they were gift card buyers over a year ago but not in the last year, means they are new HHs.

gift_hh_segment_df_older_than_2025_may = gift_hh_segment_df_old.filter(f.col("fiscal_week") < "20250404")

ehhn_in_older_mothers_day = gift_hh_segment_df_older_than_2025_may.join(
    new_3p_gift_households_mothers_day.select("ehhn"), on="ehhn", how="inner"
)

# showing the last time ehhns were a gift card buyer (if they ever were) .. should be over 52 weeks ago
display(ehhn_in_older_mothers_day)

In [0]:
gift_hh_segment_df_older_than_2025_june = gift_hh_segment_df_old.filter(f.col("fiscal_week") < "20250504")

ehhn_in_older_fathers_day = gift_hh_segment_df_older_than_2025_june.join(
    new_3p_gift_households_fathers_day.select("ehhn"), on="ehhn", how="inner"
)

# showing the last time ehhns were a gift card buyer (if they ever were) .. should be over 52 weeks ago
display(ehhn_in_older_fathers_day)

In [0]:
# ACDS transactions table
acds_sample = ACDS(use_sample_mart = False)
start = (datetime.today() - relativedelta(months=24)).strftime('%Y%m%d')
end = datetime.today().strftime('%Y%m%d')
acds_transactions = acds_sample.get_transactions(start_date = start, end_date = end, join_with = ["products", "households"])

In [0]:
# Filter to ACDS transactions between 20250604 and 20250704 (this time anchor will be different depending on the timeframe we choose above)
# Filter to ONLY HHs who are newly converted 3P Gift Buyers. We will compare this with HHs in this timeframe who are NOT gift buyers

acds_transactions_per_ehhn_per_date_mothers = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

acds_transactions_per_ehhn_per_date_filtered_mothers = acds_transactions_per_ehhn_per_date_mothers.filter(
    (f.col("ehhn").isin(new_3p_gift_households_mothers_day.select("ehhn"))) &
    # P4 actual dates
    (f.col("trn_dt").between("20250426", "20250523"))
)

acds_transactions_per_ehhn_per_date_filtered_mothers.display()

In [0]:
acds_transactions_per_ehhn_per_date_fathers = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

acds_transactions_per_ehhn_per_date_filtered_fathers = acds_transactions_per_ehhn_per_date_fathers.filter(
    (f.col("ehhn").isin(new_3p_gift_households_fathers_day.select("ehhn"))) &
    # Covers fathers day and summer
    (f.col("trn_dt").between("20250524", "20250623"))
)

acds_transactions_per_ehhn_per_date_filtered_fathers.display()

In [0]:
# Grouped HHs by Avg spend per trip for only HHs who became 3p gift buyers in May
avg_spend_per_date_per_hh_soon_to_be_gift_buyer_mothers_day = acds_transactions_per_ehhn_per_date_filtered_mothers.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_soon_to_be_gift_buyer_mothers_day)

In [0]:
# Checking variance and avg of soon to be gift buyers on Mothers day 
variance_soon_to_be_gift_buyers_mothers = (
    avg_spend_per_date_per_hh_soon_to_be_gift_buyer_mothers_day.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_soon_to_be_gift_buyers_mothers.display()

In [0]:
# Grouped HHs by Avg spend per trip for only HHs who became 3p gift buyers in May
avg_spend_per_date_per_hh_soon_to_be_gift_buyer_fathers_day = acds_transactions_per_ehhn_per_date_filtered_fathers.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_soon_to_be_gift_buyer_fathers_day)

In [0]:
# Checking variance and avg of soon to be gift buyers on FATHERS day / Grad
variance_soon_to_be_gift_buyers_fathers = (
    avg_spend_per_date_per_hh_soon_to_be_gift_buyer_fathers_day.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_soon_to_be_gift_buyers_fathers.display()

In [0]:
# Filter to ACDS transactions between 20250404 and 20250504 (this time anchor will be different depending on the timeframe we choose above)
# Filter to HHs who will NOT become 3p gift buyers or are not in gift_hh_segment_df_old
# TLDR: Control Group. Remove all current gift card shoppers as well as HHs who will become a gift card shopper

acds_transactions_per_ehhn_per_date = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

# these are HHs who are going to become 3p gift buyers in 20250504 (as a result of their habits in 20250404), as well as HHs who are already gift card buyers
relevant_gift_hh_segment_df_old_mothers = gift_hh_segment_df_old.filter(f.col("fiscal_week") >= "20240426") 
ehhn_exclude = new_3p_gift_households_mothers_day.select("ehhn").union(relevant_gift_hh_segment_df_old_mothers.select("ehhn")).distinct()

acds_transactions_per_ehhn_per_date_filtered_mothers = acds_transactions_per_ehhn_per_date.join(
    ehhn_exclude, on="ehhn", how="left_anti"
).filter(
    (f.col("trn_dt") >= "20250426") & (f.col("trn_dt") <= "20250523") & (f.col("ehhn").isNotNull())
)

acds_transactions_per_ehhn_per_date_filtered_mothers.display()

In [0]:
# Do the same here .. make July (fathers/grad day) control

acds_transactions_per_ehhn_per_date = acds_transactions.groupBy("ehhn", "trn_dt")          \
    .agg(f.round(f.sum(f.col("net_spend_amt")), 2).alias("total_spend_per_date")).orderBy("ehhn", "trn_dt")

# these are HHs who are going to become 3p gift buyers in July (as a result of their habits in June), as well as HHs who are already gift card buyers
relevant_gift_hh_segment_df_old_fathers = gift_hh_segment_df_old.filter(f.col("fiscal_week") >= "20240524") 
ehhn_exclude = new_3p_gift_households_fathers_day.select("ehhn").union(relevant_gift_hh_segment_df_old_fathers.select("ehhn")).distinct()

acds_transactions_per_ehhn_per_date_filtered_fathers = acds_transactions_per_ehhn_per_date.join(
    ehhn_exclude, on="ehhn", how="left_anti"
).filter(
    (f.col("trn_dt") >= "20250524") & (f.col("trn_dt") <= "20250623") & (f.col("ehhn").isNotNull()) # Fathers day and summer time
)

acds_transactions_per_ehhn_per_date_filtered_fathers.display()

In [0]:
# Avg spend per trip of non gift buyer in mothers day .. this might take long if sample mart is false
avg_spend_per_date_per_hh_not_gift_buyer_mothers_day = acds_transactions_per_ehhn_per_date_filtered_mothers.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_not_gift_buyer_mothers_day)

In [0]:
# Avg spend per trip of non gift buyer in fathers day
avg_spend_per_date_per_hh_not_gift_buyer_fathers_day = acds_transactions_per_ehhn_per_date_filtered_fathers.groupBy("ehhn") \
    .agg(f.avg(f.col("total_spend_per_date")).alias("avg_spend_per_date"))

display(avg_spend_per_date_per_hh_not_gift_buyer_fathers_day)

In [0]:
# Checking variance 
variance_non_gift_buyer_mothers_day = (
    avg_spend_per_date_per_hh_not_gift_buyer_mothers_day.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_non_gift_buyer_mothers_day.display()

In [0]:
# Checking variance 
variance_non_gift_buyer_fathers_day = (
    avg_spend_per_date_per_hh_not_gift_buyer_fathers_day.agg(
        f.avg("avg_spend_per_date").alias("mean_spend"),
        f.variance("avg_spend_per_date").alias("variance_spend"),
        f.stddev("avg_spend_per_date").alias("stddev_spend")
    )
)
variance_non_gift_buyer_fathers_day.display()

In [0]:
# Parametric test to determine: is there a difference in spending between HHs who will become 3p gift buyers and HHs who will not
# during peak KPF seasons? (We are testing with Holiday as per 20251204 - 20260104, also do for valentines day, mothers/fathers day)

# May Gifting Season / Mothers Day: soon to be 3p gift VS. non gift buyer 
gift_buyer_spend_A = avg_spend_per_date_per_hh_soon_to_be_gift_buyer_mothers_day.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]
nongift_buyer_spend_B = avg_spend_per_date_per_hh_not_gift_buyer_mothers_day.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]

t_stat, p_val = stats.ttest_ind(gift_buyer_spend_A, nongift_buyer_spend_B, equal_var=False)

mean_test = np.mean(gift_buyer_spend_A)
mean_control = np.mean(nongift_buyer_spend_B)
raw_lift = mean_test - mean_control

# Using 95% CI
compare_means_gift_buyer_vs_non_gift_buyer = CompareMeans(DescrStatsW(gift_buyer_spend_A), DescrStatsW(nongift_buyer_spend_B))
lower_ci, upper_ci = compare_means_gift_buyer_vs_non_gift_buyer.tconfint_diff(alpha=0.05, usevar='unequal')

print(
    f"Test Mean Spend: ${mean_test:.2f} | "
    f"Control Mean Spend: ${mean_control:.2f} | "
    f"Observed Lift: {raw_lift:+.2f} per trip | "
    f"P-value: {p_val:.7f} | "
    f"95% Confidence Interval for Lift: [${lower_ci:.2f}, ${upper_ci:.2f}]"
)

# if p-value is less than 0.05 then difference in spend between soon-to-be gift buyer and non-gift buyer in time window is stat sig


In [0]:
# JUNE Gifting Season / Fathers Day: soon to be 3p gift VS. non gift buyer 
gift_buyer_spend_A = avg_spend_per_date_per_hh_soon_to_be_gift_buyer_fathers_day.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]
nongift_buyer_spend_B = avg_spend_per_date_per_hh_not_gift_buyer_fathers_day.select("avg_spend_per_date").toPandas()["avg_spend_per_date"]

t_stat, p_val = stats.ttest_ind(gift_buyer_spend_A, nongift_buyer_spend_B, equal_var=False)

mean_test = np.mean(gift_buyer_spend_A)
mean_control = np.mean(nongift_buyer_spend_B)
raw_lift = mean_test - mean_control

# Using 95% CI
compare_means_gift_buyer_vs_non_gift_buyer = CompareMeans(DescrStatsW(gift_buyer_spend_A), DescrStatsW(nongift_buyer_spend_B))
lower_ci, upper_ci = compare_means_gift_buyer_vs_non_gift_buyer.tconfint_diff(alpha=0.05, usevar='unequal')

print(
    f"Test Mean Spend: ${mean_test:.2f} | "
    f"Control Mean Spend: ${mean_control:.2f} | "
    f"Observed Lift: {raw_lift:+.2f} per trip | "
    f"P-value: {p_val:.7f} | "
    f"95% Confidence Interval for Lift: [${lower_ci:.2f}, ${upper_ci:.2f}]"
)

# if p-value is less than 0.05 then difference in spend between soon-to-be gift buyer and non-gift buyer in time window is stat sig

### Greeting Card and Gift Card Buyers
- Capturing the relationship between Gift Card Buyers who also buy greeting cards and its inverse
- Exploring Greeting Card Frequency related data like: # of Greeting Cards / # of Trips, # of Greeting Cards bought in the last year, # of greeting cards bought in certain time periods (such as kpf promoted seasonal events) to use as features

In [0]:
# KPI session 
use_sample_mart = True
acds = ACDS(use_sample_mart = use_sample_mart)
kpi_session = KPI(use_sample_mart = use_sample_mart)
pls_data_store_session = pls_data_store(spark)

In [0]:
start_date = "2025-01-01"
end_date = "2026-01-31"

# Greeting Card Buyers + aggregated sales for a specific start-end date
sales_kpi_greeting_card = kpi_session.get_aggregate(
    start_date=start_date, end_date=end_date,   
    group_by=["ehhn"], metrics=["sales"],
    join_with=["products", "stores"], apply_golden_rules=golden_rules(["store_exclusions", "customer_exclusions"]), 
    query_filters=[
        "pid_fyt_com_cd IN ('235')",
        "mgt_div_no in ('011','014','016','018','021','024','025','026','029','034','035','531','534','615','620','660','701','703','705','706')",
        "ehhn is not null",
        "net_spend_amt >= 0",
        "scn_unt_qy >= 0"
    ]
)

sales_kpi_greeting_filtered = sales_kpi_greeting_card.filter(f.col("sales") > 0).withColumnRenamed("sales", "greeting_card_sales")
display(sales_kpi_greeting_filtered)

# Count of distinct greeting card buyers: Use to see what % of greeting card buyers are gift card buyers and vise versa
distinct_ehhn_count_greet = sales_kpi_greeting_filtered.select("ehhn").distinct().count()
print(f"Distinct ehhn count of greeting card buyers: {distinct_ehhn_count_greet}")

In [0]:
# Gift Card Buyers + aggregated sales for a specific start-end date
sales_kpi_gift_card = kpi_session.get_aggregate(
    start_date = start_date, end_date = end_date,   
    group_by=["ehhn"], metrics=["sales"],
    join_with=["products", "stores"], apply_golden_rules=golden_rules(["store_exclusions", "customer_exclusions"]), 
    query_filters=[
        "pid_fyt_com_cd IN ('300')",
        "pid_fyt_sub_com_cd not in ('30027')",
        "mgt_div_no in ('011','014','016','018','021','024','025','026','029','034','035','531','534','615','620','660','701','703','705','706')",
        "ehhn is not null",
        "net_spend_amt >= 0",
        "scn_unt_qy >= 0"
    ]
)

sales_kpi_gift_card_filtered = sales_kpi_gift_card.filter(f.col("sales") > 0).withColumnRenamed("sales", "gift_card_sales")
sales_kpi_gift_card_filtered.display()

# Count of distinct gift card buyers: Use to see what % of gift card buyers are gift card buyers and vise versa
distinct_ehhn_count_gift = sales_kpi_gift_card_filtered.select("ehhn").distinct().count()
print(f"Distinct ehhn count of gift card buyers: {distinct_ehhn_count_gift}")

In [0]:
# What % of Greeting Card Buyers bought both gift and greeting, vise versa in 2025
joined_sales = sales_kpi_greeting_filtered.join(
    sales_kpi_gift_card_filtered, on="ehhn", how="inner"
)

distinct_ehhn_count = joined_sales.select("ehhn").distinct().count()
print(f"Distinct ehhn count of greeting card buyers who also bought gift: {distinct_ehhn_count}")

percent_of_gift_who_bought_both = distinct_ehhn_count / distinct_ehhn_count_gift
percent_of_greeting_who_bought_both = distinct_ehhn_count / distinct_ehhn_count_greet
print(percent_of_gift_who_bought_both, percent_of_greeting_who_bought_both)

# 64% of Gift also buy greeting, 40% for the inverse. A bit down from 2024 but trend still holds

In [0]:
# Greeting Card Transactions 
acds_sample = ACDS(use_sample_mart = True)

acds_transactions_greeting_cards = acds_sample.get_transactions(
  start_date = start_date, end_date = end_date, join_with = ["products", "households"],
  apply_golden_rules=golden_rules(["store_exclusions", "customer_exclusions"]),
  query_filters= ["pid_fyt_com_cd IN ('235')",
        "mgt_div_no in ('011','014','016','018','021','024','025','026','029','034','035','531','534','615','620','660','701','703','705','706')",
        "ehhn is not null",
        "net_spend_amt >= 0",
        "scn_unt_qy >= 0"]
)
acds_transactions_greeting_cards.display()

In [0]:
# Avg days between greeting card transactions for EHHNs (can be a good feature, repeat HHs)
# if a HH only went once, made a flag to represent that (maybe not the type of HH we should be looking-alike)
# used trn_time as well as trn_dt to deduplicate multiple gift card purchases on the same day

acds_trips_distinct = acds_transactions.groupBy("ehhn", "trn_dt").count()
window_spec = Window.partitionBy("ehhn").orderBy("trn_dt")

acds_trips_with_lag = (
    acds_trips_distinct
    .withColumn("prev_trn_dt", f.lag("trn_dt").over(window_spec)).withColumn(
        "diff_days", f.datediff(f.to_date(f.col("trn_dt"), "yyyyMMdd"), f.to_date(f.col("prev_trn_dt"), "yyyyMMdd"))
    )
)

avg_diff_df = (
    acds_trips_with_lag.groupBy("ehhn").agg(
      f.avg("diff_days").alias("avg_days_between_trips"),
      f.when(f.count("diff_days") == 0, True).otherwise(False).alias("is_single_trip_hh")
    )
)

display(avg_diff_df)

In [0]:
# analyze avg days between trips involving a greeting card and avg spendings on greeting cards of soon-to-be gift card HHs VS. the control (Not a gift card buyer OR someone about to be a gift card buyer) 
# Lets do Valentines Day timeframe this time. In the actual model, ill probably have some greeting card related features, and seasonality related features. If results ^ are stat. sig, there would be some correlation between features without me giving it an explicit timeframe.

In [0]:
# Re-using some code for holiday gifting analysis - but this time i will only focus on greeting card transactions

# Look at Feb 2025 transactions (period 1) - Vtines day
best_baseball_segments = ["ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS"]

gift_hh_segment_df_old_2025_march = gift_hh_segment_df_old.filter(
    (f.col("fiscal_week") == "20250204") & (f.col("gcs_seg_desc").isin(best_baseball_segments))
).select("ehhn", "gcs_seg_desc")

gift_hh_segment_df_old_2025_feb = gift_hh_segment_df_old.filter(
    f.col("fiscal_week") == "20250104"
).select("ehhn")

new_3p_gift_households_after_vtines = gift_hh_segment_df_old_2025_march.join(
    gift_hh_segment_df_old_2025_feb, on="ehhn", how="left_anti"
).dropDuplicates(["ehhn"])

display(new_3p_gift_households_after_vtines)

In [0]:
# days between trips and single trip HH flag for all HHs who became 3p gift buyer in march (first time buying gift cards in feb)
greeting_card_buyers_avg_days_between_trips = (
    f.broadcast(new_3p_gift_households_after_vtines)
    .join(avg_diff_df, on="ehhn", how="inner")
)

display(greeting_card_buyers_avg_days_between_trips)

In [0]:
greeting_card_buyers_avg_days_between_trips_filtered = greeting_card_buyers_avg_days_between_trips.filter(~f.col("is_single_trip_hh"))

# avg days between greeting card purchases per baseball seg 
avg_days_between_trips_per_seg = greeting_card_buyers_avg_days_between_trips_filtered.groupBy("gcs_seg_desc").agg(
    f.avg("avg_days_between_trips")
)
display(avg_days_between_trips_per_seg)

# no grouping by
avg_days_between_trips_total = greeting_card_buyers_avg_days_between_trips_filtered.agg(
    f.avg("avg_days_between_trips"))
display(avg_days_between_trips_total)

In [0]:
# HHs who did not convert to gift card HH in timeframe (march)
test_id = new_3p_gift_households_after_vtines.select("ehhn").distinct()

# remove any hh who has touched a gift card in the last 52 wks + hhs soon to be new 3p gift hhs 
relevant_gift_hh_segment_df_old_feb = gift_hh_segment_df_old.filter(f.col("fiscal_week") >= "20240104")
ehhn_exclude = test_id.union(relevant_gift_hh_segment_df_old_feb.select("ehhn")).distinct()

sales_kpi_greeting_no_gift_buyers = (
    sales_kpi_greeting_filtered.join(ehhn_exclude, on="ehhn", how="left_anti")
)

# This will be all HHs who bought greeting cards who are not gift buyers
display(sales_kpi_greeting_no_gift_buyers)

In [0]:
# days between trips and single trip HH flag for all HHs who became 3p gift buyer in march (first time buying gift cards in feb)
greeting_card_buyers_only_avg_days_between_trips = (
    sales_kpi_greeting_no_gift_buyers
    .join(avg_diff_df, on="ehhn", how="inner")
)

display(greeting_card_buyers_only_avg_days_between_trips)

In [0]:
# Wat % of HHs buy greeting cards once and not reoccuring
distinct_ehhn_count = greeting_card_buyers_only_avg_days_between_trips.select("ehhn").distinct().count()
single_trip_hh_count = greeting_card_buyers_only_avg_days_between_trips.filter(f.col("is_single_trip_hh") == True).count()

count_of_one_time_greeting_card_buyers = single_trip_hh_count / distinct_ehhn_count
print(count_of_one_time_greeting_card_buyers)

In [0]:
# avg days between greeting card $s of non gift HHs
only_greeting_card_buyers_avg_days_between_trips_filtered = greeting_card_buyers_only_avg_days_between_trips.filter(~f.col("is_single_trip_hh"))

avg_days_between_trips_only_greet = only_greeting_card_buyers_avg_days_between_trips_filtered.agg(
    f.avg("avg_days_between_trips")
)
display(avg_days_between_trips_only_greet)

### Fuel Point Redemption + Earn Amount

In [0]:
# p11 start and end date
start_date = "2025-11-08"
end_date = "2025-12-05"

In [0]:
best_baseball_segments = ["ALL STARS", "SILVER SLUGGERS", "HOME RUN HITTERS", "AVERAGE BATTERS"]

# 2025 P11 and P12 to test during Holiday 4x Offer timeframe (Nov - Dec)
gift_hh_segment_df_old_dec = gift_hh_segment_df_old.filter(
    (f.col("fiscal_week") == "20251204") & (f.col("gcs_seg_desc").isin(best_baseball_segments))
).select("ehhn", "gcs_seg_desc")

gift_hh_segment_df_old_nov = gift_hh_segment_df_old.filter(
    f.col("fiscal_week") == "20251104"
).select("ehhn")

new_3p_gift_households = gift_hh_segment_df_old_dec.join(
    gift_hh_segment_df_old_nov, on="ehhn", how="left_anti"
).dropDuplicates(["ehhn"])

display(new_3p_gift_households)

In [0]:
# fuel points table
pls_data_store_session = pls_data_store(spark)
fuel_table_nov_dec = pls_data_store_session.get_points_detail(
  start_date=start_date, end_date=end_date, query_filters=["type == 'FUEL'"])
fuel_table_nov_dec_filtered = fuel_table_nov_dec.filter(f.col("points_earned") > 0)
fuel_table_nov_dec_filtered.display()

# points_redeemed is 0

In [0]:
#points_detail = (
#    spark.read.parquet(f'abfss://data@sa8451kemprd.dfs.core.windows.net/pls_points_v2/')
#    .filter(
#        (f.col("trn_dt") >= start_date_dt.strftime("%Y%m%d")) &
#        (f.col("trn_dt") <= end_date_dt.strftime("%Y%m%d"))
#    )
#)
#points_detail.display()

In [0]:
# all fuel point history (in p11) of HHs that became 3p gift buyers in holiday season
soon_to_be_gift_buyers_holiday_fuelpoints = new_3p_gift_households.join(fuel_table_nov_dec_filtered, on="ehhn", how="inner")
display(soon_to_be_gift_buyers_holiday_fuelpoints)

In [0]:
# total points earned per HH in P11-P12 of soon-to-be gift buyers
holiday_soon_to_be_giftbuyers_points = soon_to_be_gift_buyers_holiday_fuelpoints.groupBy("ehhn").agg(
  f.sum("points_earned").alias("total_points_earned"))
holiday_soon_to_be_giftbuyers_points.display()

In [0]:
# total points earned of HHs who are NOT soon to be gift HHs and who are NOT currently a gift HH (1 year)
# these HHs should be mutually exclusive from soon-to-be gift HHs after the holiday season, as well as HHs who have been
# gift Hhs for the past year

fuel_table_nov_dec_filtered_CONTROL = fuel_table_nov_dec_filtered.join(
    new_3p_gift_households.select("ehhn"), on="ehhn", how="left_anti"
).join(
    relevant_gift_hh_segment_df_old.select("ehhn"), on="ehhn", how="left_anti"
)

fuel_table_nov_dec_filtered_CONTROL.display()

In [0]:
# total points earned per HH in P11-P12 of soon-to-be gift buyers (test HHs for simple diagnostic test)
fuel_table_nov_dec_filtered_CONTROL_points = fuel_table_nov_dec_filtered_CONTROL.groupBy("ehhn").agg(
  f.sum("points_earned").alias("total_points_earned"))
fuel_table_nov_dec_filtered_CONTROL_points.display()

In [0]:
variance_non_gift_buyer_points_earned = (
    fuel_table_nov_dec_filtered_CONTROL_points.agg(
        f.avg("total_points_earned").alias("avg_points_earned_per_hh"),
        f.variance("total_points_earned").alias("variance_points"),
        f.stddev("total_points_earned").alias("stddev_points")
    )
)
variance_non_gift_buyer_points_earned.display()

# tons of variance .. as expected

In [0]:
# log-transforming points detail of control HH (due to skew in data, most HHs are not earning that much, but some are earning a lot )
fuel_table_nov_dec_filtered_CONTROL_points_log_transformed = fuel_table_nov_dec_filtered_CONTROL_points.withColumn("log_points", f.log1p(f.col("total_points_earned")))

variance_non_gift_buyer_points_earned_transformed = (
    fuel_table_nov_dec_filtered_CONTROL_points_log_transformed.agg(
        f.avg("log_points").alias("avg_points_earned_per_hh"),
        f.variance("log_points").alias("variance_points"),
        f.stddev("log_points").alias("stddev_points")
    )
)
variance_non_gift_buyer_points_earned_transformed.display()

In [0]:
# log-transforming points detail of TEST HHs (due to skew in data, most HHs are not earning that much, but some are earning a lot )
holiday_soon_to_be_giftbuyers_points_log = holiday_soon_to_be_giftbuyers_points.withColumn("log_points", f.log1p(f.col("total_points_earned")))

variance_soon_to_be_gift_buyer_points_earned_transformed = (
    holiday_soon_to_be_giftbuyers_points_log.agg(
        f.avg("log_points").alias("avg_points_earned_per_hh"),
        f.variance("log_points").alias("variance_points"),
        f.stddev("log_points").alias("stddev_points")
    )
)
variance_soon_to_be_gift_buyer_points_earned_transformed.display()

In [0]:
# is there a sig diff in earned points between HHs who are soon-to-be gift buyers vs. HHs who are not gift buyers or will not become one?

gift_buyer_points_A = holiday_soon_to_be_giftbuyers_points_log.select("log_points").toPandas()["log_points"]
nongift_buyer_points_B = fuel_table_nov_dec_filtered_CONTROL_points_log_transformed.select("log_points").toPandas()["log_points"]

t_stat, p_val = stats.ttest_ind(gift_buyer_points_A, nongift_buyer_points_B, equal_var=False)

mean_test = np.mean(gift_buyer_points_A)
mean_control = np.mean(nongift_buyer_points_B)
raw_lift = mean_test - mean_control

# Using 95% CI
compare_means_gift_buyer_vs_non_gift_buyer = CompareMeans(DescrStatsW(gift_buyer_points_A), DescrStatsW(nongift_buyer_points_B))
lower_ci, upper_ci = compare_means_gift_buyer_vs_non_gift_buyer.tconfint_diff(alpha=0.05, usevar='unequal')

print(
    f"Test Mean Log Points: {mean_test:.2f} | "
    f"Control Mean Log Points: {mean_control:.2f} | "
    f"Observed Lift (Log): {raw_lift:+.2f} | "
    f"P-value: {p_val:.6f} | "
    f"95% Confidence Interval: [{lower_ci:.2f}, {upper_ci:.2f}]"
)

In [0]:
# To avoid data leakage in the real model, I should not tie these features with a prev. time frame. this is just for diagnostics to give me a good idea on what to expect / how the feature might perform.

# In the real model, I expect to have a feature such as: 
# fuel points earned in the last month
# lifetime total fuel points earned (?)
# total fuel redeemed (?)

### Digital Engagement Segmentation
- HHs who are highly engaged with Kroger digital platforms. Worth investigating whether these HHs are more receptive to various KPF tactics (since most KPF campaigns are digital)

In [0]:
# digital engagement seg HHs (new (new account within 52 weeks), M, H, L)
end_date = "2026-12-23"
dig_eng = seg.get_seg_for_date('digital_engagement', end_date)
dig_eng.display()

In [0]:
# I am using Holiday XCM in 2025 as an example. Im curious to see if a HHs digital segmentation has any effect on fuel points.

holiday_offers = [
    800000176164, 800000175826, 800000176170, 800000176075,
    800000176162, 800000176161, 800000175835, 800000176087
]

pls_data_store_session = pls_data_store(spark)
fuel_table_nov_dec = pls_data_store_session.get_points_detail(
  start_date=start_date, end_date=end_date, query_filters=["type == 'FUEL'"])
fuel_table_nov_dec_numeric_offer = fuel_table_nov_dec.filter(
    f.col("offer").cast("string").rlike("^[0-9]+$")
)
fuel_table_nov_dec_filtered = fuel_table_nov_dec_numeric_offer.filter(
    (f.col("points_earned") > 0) &
    (f.col("offer").isin(holiday_offers))
)
fuel_table_nov_dec_filtered.display()

In [0]:
fuel_table_dig_eng_holiday = fuel_table_nov_dec_filtered.join(dig_eng, on="ehhn", how="inner")
display(fuel_table_dig_eng_holiday)

In [0]:
fuel_table_dig_eng_holiday \
    .groupBy("dig_eng_seg_desc", "ehhn") \
    .agg(f.sum("points_earned").alias("hh_total_points")) \
    .groupBy("dig_eng_seg_desc") \
    .agg(
        f.sum("hh_total_points").alias("total_points_earned"),
        f.count("ehhn").alias("total_hhs"),
        f.avg("hh_total_points").alias("avg_points_per_hh")
    ) \
    .display()

### Customer Dimensions
- Wondering if customer dimensions can tell us something about acquisition HHs, since there is a trend of top performing card partners in the TDC program, which may tie into consumer habits

In [0]:
# Lets look at price, health, and convenience (since most top performing card partners are dining / fast food)
funlo = seg.get_seg_for_date('funlo', end_date)
cds = seg.get_seg_for_date('cds_4_hh', end_date)

# will use seg as features instead of score in the model, but score is easier to analyze
cds_relevant_dim_seg = cds.select(
  "ehhn", "price_dim_seg", "price_dim_score", "health_dim_seg", "health_dim_score", "convenience_dim_seg", "convenience_dim_score")
cds.display()

In [0]:
# reminder to self that new_3p_gift_households are all HHs that newly became 3P gift HH in Dec (due to Nov activity), and that I used these periods as diagnostics to see the interaction of this feature during Holiday xcm 

cds_dim_gift_hhs = cds_relevant_dim_seg.join(new_3p_gift_households, on="ehhn", how="inner")
display(cds_dim_gift_hhs)

# compare funlo segmentations of HHs that will become 3p gift buyers with HHs that are lapsed / not gift buyers

In [0]:
# remove any hh who has touched gift card in last 52 wks, and remove soon to be new 3p gift hhs (new_3p_gift_hhs)
relevant_gift_hh_segment_df_old = gift_hh_segment_df_old.filter(f.col("fiscal_week") >= "20241108")
ehhn_exclude = cds_dim_gift_hhs.select("ehhn").union(relevant_gift_hh_segment_df_old.select("ehhn")).distinct()

funlo_not_gift_buyers = cds_relevant_dim_seg.join(
    ehhn_exclude, on="ehhn", how="left_anti"
)

funlo_not_gift_buyers.display()

In [0]:
# new gift buyers
cds_dim_gift_hhs_avg_scores = cds_dim_gift_hhs.agg(
    f.avg("price_dim_score").alias("avg_price_dim_score"),
    f.avg("health_dim_score").alias("avg_health_dim_score"),
    f.avg("convenience_dim_score").alias("avg_convenience_dim_score")
)

seg_summary = cds_dim_gift_hhs.agg(
    # Health breakdown
    (f.count(f.when(f.col("health_dim_seg") == "H", 1)) / f.count("*")).alias("health_H_ratio"),
    (f.count(f.when(f.col("health_dim_seg") == "M", 1)) / f.count("*")).alias("health_M_ratio"),
    (f.count(f.when(f.col("health_dim_seg") == "L", 1)) / f.count("*")).alias("health_L_ratio"),
    # Price breakdown
    (f.count(f.when(f.col("price_dim_seg") == "H", 1)) / f.count("*")).alias("price_H_ratio"),
    (f.count(f.when(f.col("price_dim_seg") == "M", 1)) / f.count("*")).alias("price_M_ratio"),
    (f.count(f.when(f.col("price_dim_seg") == "L", 1)) / f.count("*")).alias("price_L_ratio"),
    # Convenience breakdown
    (f.count(f.when(f.col("convenience_dim_seg") == "H", 1)) / f.count("*")).alias("conv_H_ratio"),
    (f.count(f.when(f.col("convenience_dim_seg") == "M", 1)) / f.count("*")).alias("conv_M_ratio"),
    (f.count(f.when(f.col("convenience_dim_seg") == "L", 1)) / f.count("*")).alias("conv_L_ratio")
)

# get avg scores per funlo segment + proportion of each segment's type vs. the total of only new gift buyers
display(cds_dim_gift_hhs_avg_scores)
display(seg_summary)


In [0]:
# not gift buyers
funlo_not_gift_buyers_avg_scores = funlo_not_gift_buyers.agg(
    f.avg("price_dim_score").alias("avg_price_dim_score"),
    f.avg("health_dim_score").alias("avg_health_dim_score"),
    f.avg("convenience_dim_score").alias("avg_convenience_dim_score")
)

seg_summary_no_gift = funlo_not_gift_buyers.agg(
    # Health breakdown
    (f.count(f.when(f.col("health_dim_seg") == "H", 1)) / f.count("*")).alias("health_H_ratio"),
    (f.count(f.when(f.col("health_dim_seg") == "M", 1)) / f.count("*")).alias("health_M_ratio"),
    (f.count(f.when(f.col("health_dim_seg") == "L", 1)) / f.count("*")).alias("health_L_ratio"),
    # Price breakdown
    (f.count(f.when(f.col("price_dim_seg") == "H", 1)) / f.count("*")).alias("price_H_ratio"),
    (f.count(f.when(f.col("price_dim_seg") == "M", 1)) / f.count("*")).alias("price_M_ratio"),
    (f.count(f.when(f.col("price_dim_seg") == "L", 1)) / f.count("*")).alias("price_L_ratio"),
    # Convenience breakdown
    (f.count(f.when(f.col("convenience_dim_seg") == "H", 1)) / f.count("*")).alias("conv_H_ratio"),
    (f.count(f.when(f.col("convenience_dim_seg") == "M", 1)) / f.count("*")).alias("conv_M_ratio"),
    (f.count(f.when(f.col("convenience_dim_seg") == "L", 1)) / f.count("*")).alias("conv_L_ratio")
)

# get avg scores per funlo segment + proportion of each segment's type vs. the total of only new gift buyers
display(funlo_not_gift_buyers_avg_scores)
display(seg_summary_no_gift)

### Self Checkout-Use: To - DO
- Perc Uscan data. Apparently this was a big predictor in the legacy model science review that Kristin sent, so worth analyzing
- Apparently this doesnt exist anymore? Use terminal number for self checkouts in ACDS transactions

In [0]:
# ACDS transactions table
acds_sample = ACDS(use_sample_mart = True)
start = (datetime.today() - relativedelta(months=12)).strftime('%Y%m%d')
end = datetime.today().strftime('%Y%m%d')
acds_transactions = acds_sample.get_transactions(start_date = start_date, end_date = end, join_with = ["products", "households"])
acds_transactions.limit(5).display()

In [0]:
# Not sure where to find perc_uscan data, which was used previously by the legacy model in 2022. Feature importance plots
# from the old model said this was a big predictor, but the results of the old model said HHs the model targeted were very similar
# to HHs the current acquisition strategy targeted ... we want to target different households since the current strategy (2025-2026) is
# not working well. Maybe dont include this feature for now..

### Final Feature List + Justification: To-Do

In [0]:
# Net Spend Amt - Worked well in legacy model, easy feature to add / prune
# Transaction and buyer history (spend per trip, total trip, number trips) during most recent holiday season (biggest kpf gifting season + first test with acquisition model being done in 26 Holiday)
# Transaction and buyer history (spend per trip, total trip, number trips) during most recent easter / mothers day / fathers day / graduation seasons (kpf gifting seasons)
# Greeting Card Sales, Number of trips involving greeting cards, avg number of days between transactions involving greeting cards, Flag (are they a one time greeting card buyer or not)
# Fuel Points Redemption and Earned Amount (lifetime, last month)
# Digital Engagement Segmentation (People who interact w/ Kroger digital media .. are they more likely to interact with SSEs, TDC, PUSH, ect)
# Customer Dimensions (funlo): Price, Health, Convenience Dimensions
# More .. ask Michelle Kelleher